In [1]:
"""
Character-level transformer, following Karpathy's "Let's build GPT".
Testbed for natural gradient methods: the optimizer is external to the model
so SGD / Adam / K-FAC can be swapped without touching anything else.
"""

# !wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

"""## 1. Imports et hyperparamètres"""

import torch
import torch.nn as nn
import torch.nn.functional as Fn   # Fn, not F: F is the Fisher matrix elsewhere

batch_size = 64
block_size = 256

n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 200

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

torch.manual_seed(1337)  # reproducibility

"""## 2. Données"""

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(s):
    return [stoi[c] for c in s]


def decode(L):
    return ''.join([itos[i] for i in L])


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  # -block_size: stay in bounds
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])  # shifted by 1: the targets
    x, y = x.to(device), y.to(device)  # On the GPU
    return x, y

"""## 3. Briques du transformeur

### 3.1 `Head`
"""


class Head(nn.Module):
    """One head of causal self-attention."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))  # not a parameter
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5  # scale by 1/sqrt(head_size): keeps softmax out of saturation
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = Fn.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v
        return out


"""### 3.2 `MultiHeadAttention`"""


class MultiHeadAttention(nn.Module):
    """Several attention heads in parallel, concatenated and mixed by a projection."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])  # ModuleList, not a list: parameters must be tracked
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # num_heads x (B,T,head_size) -> (B,T,n_embd)
        out = self.dropout(self.proj(out))
        return out


"""### 3.3 `FeedForward`"""


class FeedForward(nn.Module):
    """Position-wise MLP: each position is processed independently."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),  # without it, the two Linear layers collapse into one
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


"""### 3.4 `Block`"""


class Block(nn.Module):
    """Transformer block: communication (attention) then computation (feed-forward)."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))  # pre-norm residual: the + keeps a direct path for the gradient
        x = x + self.ffwd(self.ln2(x))
        return x


"""## 4. Modèle"""


class GPTLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = Fn.cross_entropy(logits, targets)  # cross_entropy wants (N,C) and (N,)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = Fn.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


"""## 5. Évaluation et entraînement"""


@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train(model, optimizer, max_iters=max_iters, eval_interval=eval_interval, verbose=True):
    """The optimizer is passed in, not built here: the same training loop is
    reused for SGD, Adam and the natural gradient, so the comparison is fair.
    Returns the history of evaluations for later plotting."""
    import time
    history = []
    t0 = time.time()

    for it in range(max_iters):
        if it % eval_interval == 0 or it == max_iters - 1:
            losses = estimate_loss(model)
            history.append({'step': it,
                            'train': losses['train'],
                            'val': losses['val'],
                            'time': time.time() - t0})
            if verbose:
                print(f"step {it:5d} | train {losses['train']:.4f} | "
                      f"val {losses['val']:.4f} | {time.time()-t0:6.1f}s")

        xb, yb = get_batch('train')
        _, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    return history


"""### 5.1 Lancement"""

model = GPTLanguageModel(vocab_size).to(device)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M paramètres")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
history_adamw = train(model, optimizer)

# save the trained weights so a rerun does not pay the training time again
torch.save(model.state_dict(), 'mini_gpt_adamw.pt')
# model.load_state_dict(torch.load('mini_gpt_adamw.pt'))

"""## 6. Génération"""

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=2000)[0].tolist()))

cpu


FileNotFoundError: [Errno 2] No such file or directory: 'input.txt'